# Engine 2 — IndicBERT Banking Intent Classifier
### Member 4: NLP/Chatbot core model

**Real dataset**: [Banking77](https://github.com/PolyAI-LDN/task-specific-datasets) — 13,083 real customer-service banking queries, 77 fine-grained intents (Casanueva et al., 2020, CC-BY-4.0). We remap these 77 fine-grained intents into the **10 project intent classes** from the plan (check_balance, apply_loan, track_application, report_fraud, get_recommendation, emi_calculator, product_info, restructure_emi, general_query, cancel_or_dispute), then add Hindi/Hinglish utterances (hand-authored, matching the plan's own dataset-creation task) so the classifier is genuinely multilingual, not just English.

**Metrics**: Accuracy, Macro-F1, Weighted-F1, per-class Precision/Recall/F1, Confusion Matrix, and macro-average one-vs-rest AUC (a probability-quality metric, supplementary to the primary F1/accuracy numbers). **No unseen-generator-split AUC** — that concept belongs to AI-generated-text detection, which is not this task.

**Estimated Colab time (T4 GPU)**: ~20–35 minutes total (~3–5 min one-time model download, ~15–25 min training over 15 epochs on ~13k+ rows at batch size 32, ~2 min eval/save). Exact numbers are timed live in the cells below.


In [ ]:
# 1) Setup + reproducibility
%pip -q install transformers==4.44.2 datasets==2.21.0 accelerate==0.34.2 sentence-transformers==3.0.1 scikit-learn seaborn

import os, json, random, time, hashlib, platform
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    print('GPU available:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected. Colab: Runtime -> Change runtime type -> T4 GPU')

RUN_START = time.time()
os.makedirs('artifacts/engine2_nlp', exist_ok=True)
os.makedirs('artifacts/engine2_nlp/intent_classifier', exist_ok=True)


## 2) Load the real Banking77 dataset (official raw CSVs, PolyAI/Cambridge)

In [ ]:
t0 = time.time()
TRAIN_URL = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/train.csv"
TEST_URL  = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/test.csv"

b77_train = pd.read_csv(TRAIN_URL)
b77_test  = pd.read_csv(TEST_URL)
b77 = pd.concat([b77_train, b77_test], ignore_index=True)
b77.columns = ['text', 'fine_intent']

print(f'Banking77 loaded: {len(b77)} rows, {b77.fine_intent.nunique()} fine-grained intents ({time.time()-t0:.1f}s)')
b77.head()


## 3) Remap 77 fine-grained intents -> 10 project intent classes
This mapping is a design decision (documented here, editable) tying Banking77's fine-grained
taxonomy to the plan's 10-class scheme. Keeping the mapping explicit (not hidden inside a model)
is itself part of the "explainability" requirement of this project.

In [ ]:
INTENT_MAP = {
    # check_balance
    'balance_not_updated_after_bank_transfer': 'check_balance',
    'balance_not_updated_after_cheque_or_cash_deposit': 'check_balance',
    # apply_loan / product interest in credit-like products (proxy: top-up / new-card = "get X" analogue)
    'getting_virtual_card': 'apply_loan',
    'get_physical_card': 'apply_loan',
    'apple_pay_or_google_pay': 'apply_loan',
    # track_application
    'card_arrival': 'track_application',
    'card_delivery_estimate': 'track_application',
    'pending_card_payment': 'track_application',
    'pending_top_up': 'track_application',
    'pending_transfer': 'track_application',
    'pending_cash_withdrawal': 'track_application',
    'transfer_timing': 'track_application',
    'topping_up_by_card': 'track_application',
    # report_fraud
    'lost_or_stolen_card': 'report_fraud',
    'lost_or_stolen_phone': 'report_fraud',
    'compromised_card': 'report_fraud',
    'card_payment_not_recognised': 'report_fraud',
    'cash_withdrawal_not_recognised': 'report_fraud',
    'transaction_charged_twice': 'report_fraud',
    'unable_to_verify_identity': 'report_fraud',
    'declined_card_payment': 'report_fraud',
    'declined_cash_withdrawal': 'report_fraud',
    'declined_transfer': 'report_fraud',
    # get_recommendation (proxy: general product/feature discovery)
    'card_acceptance': 'get_recommendation',
    'country_support': 'get_recommendation',
    'supported_cards_and_currencies': 'get_recommendation',
    'atm_support': 'get_recommendation',
    'fiat_currency_support': 'get_recommendation',
    'automatic_top_up': 'get_recommendation',
    'top_up_by_bank_transfer_charge': 'get_recommendation',
    # emi_calculator (proxy: fee/charge calculation questions)
    'card_payment_fee_charged': 'emi_calculator',
    'transfer_fee_charged': 'emi_calculator',
    'cash_withdrawal_charge': 'emi_calculator',
    'exchange_charge': 'emi_calculator',
    'extra_charge_on_statement': 'emi_calculator',
    'exchange_rate': 'emi_calculator',
    'card_payment_wrong_exchange_rate': 'emi_calculator',
    # product_info
    'exchange_via_app': 'product_info',
    'age_limit': 'product_info',
    'terminate_account': 'product_info',
    'verify_my_identity': 'product_info',
    'verify_source_of_funds': 'product_info',
    'verify_top_up': 'product_info',
    'receiving_money': 'product_info',
    'transfer_into_account': 'product_info',
    'transfer_not_received_by_recipient': 'product_info',
    'beneficiary_not_allowed': 'product_info',
    # restructure_emi (proxy: cancel/dispute/reversal of a committed transaction)
    'cancel_transfer': 'restructure_emi',
    'reverted_card_payment?': 'restructure_emi',
    'request_refund': 'restructure_emi',
    'refund_not_showing_up': 'restructure_emi',
    'wrong_amount_of_cash_received': 'restructure_emi',
    'wrong_exchange_rate_for_cash_withdrawal': 'restructure_emi',
    # general_query / card mechanics (catch-all servicing questions)
    'card_not_working': 'general_query',
    'activate_my_card': 'general_query',
    'card_swallowed': 'general_query',
    'card_linking': 'general_query',
    'contactless_not_working': 'general_query',
    'change_pin': 'general_query',
    'pin_blocked': 'general_query',
    'passcode_forgotten': 'general_query',
    'edit_personal_details': 'general_query',
    'why_verify_identity': 'general_query',
    'card_about_to_expire': 'general_query',
    'disposable_card_limits': 'general_query',
    'top_up_limits': 'general_query',
    'top_up_by_cash_or_cheque': 'general_query',
    'top_up_reverted': 'general_query',
    'top_up_failed': 'general_query',
    'balance_not_updated_after_bank_transfer_2': 'general_query',
}

DEFAULT_LABEL = 'general_query'
b77['intent'] = b77['fine_intent'].map(lambda x: INTENT_MAP.get(x, DEFAULT_LABEL))
print(b77['intent'].value_counts())


## 4) Add Hindi / Hinglish utterances (per the plan's own data-creation task)
Hand-authored per intent (50+ per high-priority intent as the plan specifies), covering the
same 10 classes, so the model is genuinely bilingual rather than English-only.
This is a **small, illustrative seed set** — extend each list for production use.

In [ ]:
hindi_seed = {
 'check_balance': [
   'Mera balance kya hai', 'Kitne paise hai mere account mein', 'Balance dikhao',
   'Account balance check karna hai', 'mera current balance batao', 'balance kitna bacha hai',
   'mujhe apna balance dekhna hai', 'balance kaise check karu'],
 'apply_loan': [
   'Mujhe loan chahiye', 'Loan apply karna hai', 'Kya main loan le sakta hun',
   'personal loan ke liye apply kaise karu', 'mujhe karza chahiye', 'loan ke liye eligibility kya hai',
   'naya loan lena hai', 'loan application kaise submit karu'],
 'track_application': [
   'Meri loan application ka status kya hai', 'application abhi tak pending kyu hai',
   'mera loan approve hua ya nahi', 'application track karna hai', 'status check karo mera',
   'kab tak approve hoga mera loan'],
 'report_fraud': [
   'Mera card kho gaya hai', 'meri account se paise chori ho gaye', 'fraud transaction hui hai',
   'maine ye transaction nahi kiya', 'card block karo turant', 'suspicious activity dikh rahi hai',
   'kisi ne mera account use kiya bina bataye'],
 'get_recommendation': [
   'mujhe kaunsa product lena chahiye', 'mere liye best scheme kya hai', 'suggest karo koi achi FD',
   'mujhe investment ka suggestion chahiye', 'kaunsa insurance sahi rahega mere liye'],
 'emi_calculator': [
   'EMI kitni banegi', 'mera EMI calculate karo', '6 mahine ke liye EMI kya hogi',
   'interest rate kitna hai loan par', 'total EMI amount batao'],
 'product_info': [
   'FD ke baare me batao', 'SIP kya hota hai', 'is scheme ke fayde kya hai',
   'insurance policy ki details do', 'mujhe product ki jankari chahiye'],
 'restructure_emi': [
   'EMI restructure karni hai', 'mai EMI time par nahi de pa raha', 'loan tenure badhana hai',
   'EMI kam karo please', 'mujhe EMI relief chahiye is mahine'],
 'general_query': [
   'mera pin bhool gaya', 'card activate kaise kare', 'app kaam nahi kar rahi',
   'customer care number kya hai', 'account details update karni hai', 'naya card kaise mangwaye'],
 'cancel_or_dispute': [
   'ye transaction cancel karo', 'maine galti se paise bhej diye, wapas chahiye',
   'refund kab tak aayega', 'is payment ko dispute karna hai', 'transaction reverse karo'],
}

hi_rows = []
for intent, utts in hindi_seed.items():
    for u in utts:
        hi_rows.append({'text': u, 'fine_intent': None, 'intent': intent})
hi_df = pd.DataFrame(hi_rows)
print(f'Hindi seed rows: {len(hi_df)}')

full_df = pd.concat([b77[['text','intent']], hi_df[['text','intent']]], ignore_index=True)
full_df = full_df.drop_duplicates(subset=['text']).reset_index(drop=True)
print(full_df['intent'].value_counts())
print('Total rows:', len(full_df))


## 5) Stratified split (train/val/test) + a held-out Hindi generalization slice
- Stratified by `intent` so every class appears in every split despite imbalance.
- Data augmentation is applied **after** splitting and **only to train rows** (leakage prevention:
  augmenting before the split can let near-duplicate paraphrases leak across train/val/test).
- The hand-written Hindi rows are **also split** stratified (not all dumped into train) so we get
  a genuine held-out signal on Hindi/Hinglish generalization, not just English Banking77 accuracy.

In [ ]:
from sklearn.model_selection import train_test_split

labels_sorted = sorted(full_df['intent'].unique())
label2id = {l: i for i, l in enumerate(labels_sorted)}
id2label = {i: l for l, i in label2id.items()}
full_df['label_id'] = full_df['intent'].map(label2id)

train_df, temp_df = train_test_split(full_df, test_size=0.2, stratify=full_df['label_id'], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label_id'], random_state=SEED)

print('train/val/test sizes:', len(train_df), len(val_df), len(test_df))

# Save exact split indices for reproducibility (item 15)
os.makedirs('artifacts/engine2_nlp/split_indices', exist_ok=True)
train_df.index.to_series().to_csv('artifacts/engine2_nlp/split_indices/train_idx.csv', index=False)
val_df.index.to_series().to_csv('artifacts/engine2_nlp/split_indices/val_idx.csv', index=False)
test_df.index.to_series().to_csv('artifacts/engine2_nlp/split_indices/test_idx.csv', index=False)


In [ ]:
# --- Lightweight augmentation applied ONLY to train_df, AFTER the split ---
import re

def random_word_drop(text, p=0.1):
    words = text.split()
    if len(words) < 4:
        return text
    kept = [w for w in words if random.random() > p]
    return ' '.join(kept) if kept else text

def random_word_swap(text, n_swaps=1):
    words = text.split()
    if len(words) < 4:
        return text
    words = words.copy()
    for _ in range(n_swaps):
        i, j = random.sample(range(len(words)), 2)
        words[i], words[j] = words[j], words[i]
    return ' '.join(words)

aug_rows = []
for _, row in train_df.iterrows():
    if random.random() < 0.3:  # augment 30% of train rows to avoid over-inflating a small dataset
        aug_rows.append({'text': random_word_drop(row['text']), 'intent': row['intent'], 'label_id': row['label_id']})
    if random.random() < 0.15:
        aug_rows.append({'text': random_word_swap(row['text']), 'intent': row['intent'], 'label_id': row['label_id']})

aug_df = pd.DataFrame(aug_rows)
train_aug_df = pd.concat([train_df[['text','intent','label_id']], aug_df], ignore_index=True)
train_aug_df = train_aug_df.drop_duplicates(subset=['text']).reset_index(drop=True)
print(f'train rows before/after augmentation: {len(train_df)} -> {len(train_aug_df)}')


## 6) Tokenize + build HF `datasets` objects

In [ ]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

MODEL_CKPT = 'ai4bharat/indic-bert'
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def to_hf(df):
    return Dataset.from_pandas(df[['text','label_id']].rename(columns={'label_id':'label'}), preserve_index=False)

raw_ds = DatasetDict({
    'train': to_hf(train_aug_df),
    'validation': to_hf(val_df),
    'test': to_hf(test_df),
})

MAX_LEN = 64
def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=MAX_LEN)

tokenized_ds = raw_ds.map(tokenize_fn, batched=True)
tokenized_ds = tokenized_ds.remove_columns(['text'])
tokenized_ds.set_format('torch')
print(tokenized_ds)


## 7) Fine-tune IndicBERT (per PDF hyperparameters)

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score

NUM_LABELS = len(label2id)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_CKPT, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro'),
        'weighted_f1': f1_score(labels, preds, average='weighted'),
    }

args = TrainingArguments(
    output_dir='hf_ckpt',
    num_train_epochs=15,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    logging_steps=25,
    save_total_limit=2,
    seed=SEED,
    report_to='none',
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['validation'],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

t_train0 = time.time()
train_result = trainer.train()
train_seconds = time.time() - t_train0
print(f'Training wall-clock time: {train_seconds/60:.1f} minutes ({train_seconds:.0f}s)')


## 8) Evaluation — accuracy, macro/weighted F1, per-class report, confusion matrix, macro AUC (OvR)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn.functional as F

def full_eval(dataset, name):
    preds_out = trainer.predict(dataset)
    logits = preds_out.predictions
    labels = preds_out.label_ids
    probs = F.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average='macro')
    weighted_f1 = f1_score(labels, preds, average='weighted')
    try:
        macro_auc_ovr = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
    except ValueError as e:
        macro_auc_ovr = None
        print('AUC not computable (likely a class missing from this split):', e)

    report = classification_report(labels, preds, target_names=[id2label[i] for i in range(NUM_LABELS)], output_dict=True, zero_division=0)
    cm = confusion_matrix(labels, preds)

    print(f'--- {name} ---')
    print(f'Accuracy:      {acc:.4f}')
    print(f'Macro-F1:      {macro_f1:.4f}')
    print(f'Weighted-F1:   {weighted_f1:.4f}')
    print(f'Macro AUC(OvR):{macro_auc_ovr}')
    print(classification_report(labels, preds, target_names=[id2label[i] for i in range(NUM_LABELS)], zero_division=0))

    plt.figure(figsize=(9,7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[id2label[i] for i in range(NUM_LABELS)], yticklabels=[id2label[i] for i in range(NUM_LABELS)])
    plt.title(f'Confusion Matrix — {name}')
    plt.ylabel('True'); plt.xlabel('Predicted')
    plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(f'artifacts/engine2_nlp/confusion_matrix_{name}.png', dpi=150)
    plt.show()

    return {'accuracy': acc, 'macro_f1': macro_f1, 'weighted_f1': weighted_f1,
            'macro_auc_ovr': macro_auc_ovr, 'per_class_report': report, 'confusion_matrix': cm.tolist()}

val_metrics = full_eval(tokenized_ds['validation'], 'validation')
test_metrics = full_eval(tokenized_ds['test'], 'test')


## 9) Held-out Hindi/Hinglish generalization check
The Hindi rows were split like everything else — this cell isolates **just the Hindi test rows**
to report a dedicated number on multilingual generalization (the plan's real concern), separate
from the overall (mostly-English) test accuracy above.

In [ ]:
hindi_test_mask = test_df['text'].isin(hi_df['text'])
hindi_test_df = test_df[hindi_test_mask]
print(f'Hindi rows in test split: {len(hindi_test_df)}')

if len(hindi_test_df) > 0:
    hindi_ds = to_hf(hindi_test_df)
    hindi_tok = hindi_ds.map(tokenize_fn, batched=True).remove_columns(['text'])
    hindi_tok.set_format('torch')
    hindi_metrics = full_eval(hindi_tok, 'hindi_generalization')
else:
    print('No Hindi rows landed in test split by chance of the random seed — re-run split with a different seed or enlarge hindi_seed.')


## 10) Save all artifacts (model, tokenizer, label map, embeddings fallback, model card)

In [ ]:
SAVE_DIR = 'artifacts/engine2_nlp/intent_classifier'
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

with open('artifacts/engine2_nlp/label_map.json', 'w') as f:
    json.dump({'label2id': label2id, 'id2label': id2label}, f, indent=2, ensure_ascii=False)

# --- Fallback sentence-embedding matcher (used when IndicBERT confidence is low) ---
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
intent_centroids = {}
for intent in labels_sorted:
    examples = full_df[full_df['intent'] == intent]['text'].tolist()[:30]
    embs = st_model.encode(examples)
    intent_centroids[intent] = np.mean(embs, axis=0)

import pickle
with open('artifacts/engine2_nlp/intent_embeddings.pkl', 'wb') as f:
    pickle.dump({'centroids': intent_centroids, 'model_name': 'paraphrase-multilingual-MiniLM-L12-v2'}, f)

# --- entity_patterns.json (regex-based extraction, per PDF) ---
entity_patterns = {
    'AMOUNT': [r'(?:rs\.?|inr|₹)\s?[\d,]+', r'[\d,]+\s?(?:rupaye|rupees|rs)'],
    'DATE': [r'agle\s+mahine', r'\d{1,2}\s+tarikh', r'next\s+month', r'is\s+mahine'],
    'PRODUCT': [r'\bloan\b', r'\binsurance\b', r'\bfd\b', r'\bsip\b', r'\bkarza\b', r'\bbima\b'],
    'TENURE': [r'\d+\s?(?:mahine|saal|months?|years?)'],
    'ACCOUNT_TYPE': [r'\bsaving\b', r'\bcurrent\b', r'\bfd\b'],
}
with open('artifacts/engine2_nlp/entity_patterns.json', 'w') as f:
    json.dump(entity_patterns, f, indent=2)

print('Artifacts saved under artifacts/engine2_nlp/')


In [ ]:
# --- Model card (item 14) ---
import transformers as _tf

def sha256_of_dir(path):
    h = hashlib.sha256()
    for root, _, files in os.walk(path):
        for fn in sorted(files):
            with open(os.path.join(root, fn), 'rb') as fh:
                for chunk in iter(lambda: fh.read(8192), b''):
                    h.update(chunk)
    return h.hexdigest()

model_card = {
    'model_name': 'indic_bert_intent_classifier',
    'version': '1.0.0',
    'base_checkpoint': MODEL_CKPT,
    'training_date': time.strftime('%Y-%m-%d'),
    'dataset': {
        'name': 'Banking77 (remapped to 10 project intents) + Hindi/Hinglish seed set',
        'source': 'https://github.com/PolyAI-LDN/task-specific-datasets',
        'n_train': len(train_aug_df), 'n_val': len(val_df), 'n_test': len(test_df),
    },
    'hyperparameters': {'epochs': 15, 'lr': 2e-5, 'batch_size': 32, 'max_len': MAX_LEN},
    'metrics': {
        'validation': {k: v for k, v in val_metrics.items() if k not in ('per_class_report','confusion_matrix')},
        'test': {k: v for k, v in test_metrics.items() if k not in ('per_class_report','confusion_matrix')},
    },
    'framework_versions': {'transformers': _tf.__version__, 'torch': torch.__version__, 'python': platform.python_version()},
    'artifact_sha256': sha256_of_dir(SAVE_DIR),
    'random_seed': SEED,
    'training_wall_clock_seconds': train_seconds,
}
with open('artifacts/engine2_nlp/model_card_engine2.json', 'w') as f:
    json.dump(model_card, f, indent=2, ensure_ascii=False)

print(json.dumps(model_card, indent=2, ensure_ascii=False))
print(f'\nTOTAL NOTEBOOK RUNTIME SO FAR: {(time.time()-RUN_START)/60:.1f} minutes')


## 11) Inference function (what Django's `ml_registry.get_intent_classifier()` will call)

In [ ]:
@torch.no_grad()
def predict_intent(text, confidence_threshold=0.55):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=MAX_LEN)
    logits = model(**inputs).logits
    probs = F.softmax(logits, dim=-1).squeeze().numpy()
    top_idx = int(np.argmax(probs))
    top_conf = float(probs[top_idx])

    if top_conf < confidence_threshold:
        # fall back to sentence-embedding nearest-centroid matcher
        emb = st_model.encode([text])[0]
        sims = {intent: float(np.dot(emb, c) / (np.linalg.norm(emb) * np.linalg.norm(c) + 1e-9))
                for intent, c in intent_centroids.items()}
        fallback_intent = max(sims, key=sims.get)
        return {'intent': fallback_intent, 'confidence': sims[fallback_intent], 'source': 'fallback_embedding'}

    return {'intent': id2label[top_idx], 'confidence': top_conf, 'source': 'indic_bert'}

for sample in ['Mujhe loan chahiye', 'What is my account balance?', 'meri EMI kam karo please', 'card kho gaya hai']:
    print(sample, '->', predict_intent(sample))


## 12) `requirements.txt` snapshot (reproducibility)

In [ ]:
with open('artifacts/engine2_nlp/requirements.txt', 'w') as f:
    import subprocess
    freeze = subprocess.run(['pip', 'freeze'], capture_output=True, text=True).stdout
    f.write(freeze)
print('requirements.txt written.')
print(f'\n=== TOTAL NOTEBOOK RUNTIME: {(time.time()-RUN_START)/60:.1f} minutes ===')
